Əla 👍 onda **“Bazar”** adlı sayt üçün **tam real-life skelet + işlək kod nümunələri** verim. Bu, **production-a gedən yolun özü**dür.

Aşağıda **backend (Django + Cloudinary + API)** və **frontend (GitHub Pages + JS)** ayrı-ayrı verilir.

---

#### 🏗️ ÜMUMİ ARXİTEKTURA (Bazar)

```
Admin (Django)
  └── şəkil upload edir
        ↓
Cloudinary (şəkli saxlayır)
        ↓
Django DB → image URL
        ↓
Django REST API (JSON)
        ↓
GitHub Pages (HTML + JS)
```

---

#### 1️⃣ Django Backend (API)

#### 📁 Project strukturu

```
bazar_backend/
 ├── bazar/
 │    ├── settings.py
 │    ├── urls.py
 ├── products/
 │    ├── models.py
 │    ├── admin.py
 │    ├── serializers.py
 │    ├── views.py
 │    ├── urls.py
 └── manage.py
```

---

#### 2️⃣ Lazım olan paketlər

```bash
pip install django djangorestframework cloudinary pillow django-cors-headers
```

---

#### 3️⃣ Cloudinary setup (`settings.py`)

```python
INSTALLED_APPS = [
    'django.contrib.admin',
    'django.contrib.auth',
    'django.contrib.contenttypes',
    'django.contrib.sessions',
    'django.contrib.messages',
    'django.contrib.staticfiles',

    'rest_framework',
    'corsheaders',
    'cloudinary',
    'cloudinary_storage',

    'products',
]

MIDDLEWARE = [
    'corsheaders.middleware.CorsMiddleware',
    'django.middleware.security.SecurityMiddleware',
    'django.contrib.sessions.middleware.SessionMiddleware',
    'django.middleware.common.CommonMiddleware',
    'django.middleware.csrf.CsrfViewMiddleware',
    'django.contrib.auth.middleware.AuthenticationMiddleware',
    'django.contrib.messages.middleware.MessageMiddleware',
]

CORS_ALLOW_ALL_ORIGINS = True  # GitHub Pages üçün

# Cloudinary
CLOUDINARY_STORAGE = {
    'CLOUD_NAME': 'cloud_name',
    'API_KEY': 'api_key',
    'API_SECRET': 'api_secret',
}

DEFAULT_FILE_STORAGE = 'cloudinary_storage.storage.MediaCloudinaryStorage'
```

📌 **Qeyd:** Bunları Cloudinary dashboard-dan götürürsən.

---

#### 4️⃣ Product modeli (`models.py`)

```python
from django.db import models

class Product(models.Model):
    title = models.CharField(max_length=200)
    price = models.DecimalField(max_digits=10, decimal_places=2)
    description = models.TextField(blank=True)
    image = models.ImageField(upload_to='bazar_products/')
    created_at = models.DateTimeField(auto_now_add=True)

    def __str__(self):
        return self.title
```

✔️ Şəkil **kompüterdə yox**, Cloudinary-də saxlanacaq

---

#### 5️⃣ Django Admin (`admin.py`)

```python
from django.contrib import admin
from .models import Product

@admin.register(Product)
class ProductAdmin(admin.ModelAdmin):
    list_display = ('title', 'price', 'created_at')
```

👉 Admin paneldən rahat upload edirsən

---

#### 6️⃣ Serializer (`serializers.py`)

```python
from rest_framework import serializers
from .models import Product

class ProductSerializer(serializers.ModelSerializer):
    image = serializers.ImageField(use_url=True)

    class Meta:
        model = Product
        fields = '__all__'
```

---

#### 7️⃣ API View (`views.py`)

```python
from rest_framework.generics import ListAPIView
from .models import Product
from .serializers import ProductSerializer

class ProductListView(ListAPIView):
    queryset = Product.objects.all().order_by('-created_at')
    serializer_class = ProductSerializer
```

---

#### 8️⃣ API URL (`products/urls.py`)

```python
from django.urls import path
from .views import ProductListView

urlpatterns = [
    path('products/', ProductListView.as_view(), name='product-list'),
]
```

#### `bazar/urls.py`

```python
from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path('admin/', admin.site.urls),
    path('api/', include('products.urls')),
]
```

---

#### 9️⃣ API nəticəsi (nümunə JSON)

```json
[
  {
    "id": 1,
    "title": "iPhone 11",
    "price": "650.00",
    "description": "Yaxşı vəziyyətdə",
    "image": "https://res.cloudinary.com/...jpg"
  }
]
```

---

#### 2️⃣ GitHub Pages (Frontend)

#### 📁 Repo strukturu

```
bazar-frontend/
 ├── index.html
 ├── style.css
 └── app.js
```

---

#### 1️⃣ `index.html`

```html
<!DOCTYPE html>
<html lang="az">
<head>
  <meta charset="UTF-8">
  <title>Bazar</title>
  <link rel="stylesheet" href="style.css">
</head>
<body>
  <h1>Bazar – İkinci əl məhsullar</h1>

  <div id="products"></div>

  <script src="app.js"></script>
</body>
</html>
```

---

#### 2️⃣ `style.css`

```css
body {
  font-family: Arial;
  background: #f5f5f5;
}

.product {
  background: white;
  padding: 15px;
  margin: 10px;
  width: 250px;
  display: inline-block;
}

.product img {
  width: 100%;
}
```

---

#### 3️⃣ `app.js`

```javascript
const searchInput = document.createElement("input");
searchInput.placeholder = "Axtar...";
document.body.prepend(searchInput);

const container = document.getElementById("products");

function loadProducts(q = "") {
  fetch(`https://bazar-backend-5cka.onrender.com/api/products/search/?q=${q}`)
    .then(res => res.json())
    .then(data => {
      container.innerHTML = "";

      if (data.length === 0) {
        container.innerHTML = "<p>Məhsul tapılmadı</p>";
        return;
      }

      data.forEach(item => {
        container.innerHTML += `
          <div class="product">
            <img src="${item.image}" />
            <h3>${item.title}</h3>
            <p>${item.description}</p>
            <b>${item.price} AZN</b>
          </div>
        `;
      });
    })
    .catch(err => {
      console.error("Xəta:", err);
    });
}

/* 🔥 SƏHİFƏ AÇILANDA AVTOMATİK YÜKLƏ */
loadProducts();

/* 🔍 AXTARIŞ */
searchInput.addEventListener("keyup", (e) => {
  loadProducts(e.target.value);
});

```

📌 Backend URL → Heroku / Railway / Render

---

#### ✅ NƏ QAZANDIN?

✔️ Şəkillər **kompüterində saxlanmır**
✔️ Limitsiz məhsul
✔️ GitHub Pages problemsiz
✔️ Admin rahat
✔️ Real marketplace strukturu

---

#### İstəsən növbəti addımlar

* 🔐 Login / JWT
* 📤 İstifadəçi özü məhsul əlavə etsin
* ❤️ Favoritlər
* 🔍 Axtarış / filter
* 📱 Mobil UI

De hansını istəyirsən, **birbaşa ora keçək** 😎


1) migrations olunub +
2) superuser admin django olunub +
3) requriemnts.txt olunub +
4) backendi  githuba yukle +
5) backendi render.com-a yukle +
6) render.com dan geleni fromtede yaz +
7) frontedi githuba
8)  sonda github io

###  davami

Bəli, sənin düşündüyün ardıcıllıq **ümumiyyətlə düzgün**dir, sadəcə addım-addım izah edim ki qarışmasın:

---

## 1️⃣ Backend hazırlığı

1. Django proyektini **GitHub**-a yüklə.

   * `.gitignore`-də `__pycache__/`, `venv/`, `db.sqlite3` kimi lazımsız fayllar olmasın.
2. Django `requirements.txt` faylını yarat:

   ```bash
   pip freeze > requirements.txt
   ```
3. `Procfile` əlavə et (Render üçün):

   ```
   web: gunicorn bazar.wsgi
   ```
4. `settings.py`-də `ALLOWED_HOSTS`-ə Render domain əlavə et:

   ```python
   ALLOWED_HOSTS = ['*.onrender.com']
   ```

---

## 2️⃣ Backend deploy

1. Render.com-da **New Web Service** seç.
2. GitHub reposunu bağla.
3. Branch seç və deploy et.
4. Deploy tamamlandıqdan sonra Render sənə **backend URL** verəcək. Məsələn:

   ```
   https://bazar-backend.onrender.com/api/products/search/?q=...
   ```

---

## 3️⃣ Frontend hazırlığı

1. `frontend/app.js` içində fetch URL-ni Render linki ilə dəyiş:

   ```javascript
   fetch(`https://bazar-backend.onrender.com/api/products/search/?q=${q}`)
   ```
2. Frontend fayllarını GitHub reposuna yüklə (`index.html`, `app.js`, `style.css` və s.).

---

## 4️⃣ Frontend deploy (GitHub Pages)

1. GitHub reposunda **Settings → Pages** bölməsində `main` branch + `/root` seç.
2. GitHub Pages sənə link verəcək, məsələn:

   ```
   https://username.github.io/bazar-frontend/
   ```
3. Frontend artıq browser-da çalışacaq və **realtime axtarış** üçün fetch Render backend-ə sorğu göndərəcək.

---

✅ Nəticə:

* Backend → Render (API URL hazır)
* Frontend → GitHub Pages (JS fetch ilə backend-ə bağlanır)
* Browser-da frontend açanda Django API-dən JSON gəlir və axtarış işləyir.

---

Əgər istəyirsənsə, mən sənə **tam hazır layihə strukturunu** göstərə bilərəm, harada nə fayl olacaq, hər şeyi deploy üçün hazır şəkildə.

Bunu hazırlayım?


### duzgun ad secimi

Aha, indi anladım. Sən istəyirsən ki, frontend-in **GitHub Pages linki qısa və sadə** olsun, məsələn:

```
https://username.github.io/bazar
```

Bunu etmək olar, amma bir neçə şərt var:

---

## 1️⃣ Repo adını düzgün seçmək

* GitHub Pages linki **repo adından** gəlir:

  * Məsələn, repo adı `bazar` olarsa:

    ```
    https://username.github.io/bazar/
    ```
* Əgər repo adı fərqli olsa, link də fərqli olacaq.

---

## 2️⃣ Branch seçimi

* Frontend üçün **main branch**-i istifadə edə bilərsən.
* GitHub Pages settings-də:

  * **Source** → `main` branch
  * **Folder** → `/root`

Bu halda bütün `index.html`, `app.js`, `style.css` əsas `main` branch-də olmalıdır.

---

## 3️⃣ Frontend fetch URL-i

* `app.js`-də `fetch` URL-i **Render backend linki** olmalıdır.

  ```javascript
  fetch('https://bazar-backend.onrender.com/api/products/search/?q=' + q)
  ```

---

### 🔹 Yekun

* Repo adı: `bazar`
* Branch: `main`
* GitHub Pages: `/root`
* Nəticə: `https://username.github.io/bazar/`

---

Əgər istəyirsənsə, mən sənə **tam layihə strukturu ilə harada nə fayl olacaq və deploy addımları** hazırlaya bilərəm ki, bir klikdə hazır olsun.

Bunu edim?


### Şəkil problemi 

Əlbəttə 👍 **eyni məzmunu təmiz, səliqəli formada yenidən çap edirəm**. Heç nə dəyişmir, bu **tam düzgün production yoludur**. Sakit-sakit get 👇

---

#### 🔵 1. Cloudinary hesabı yarat

1️⃣ Gir: **cloudinary.com**
2️⃣ Sign up (Google ilə də olar)
3️⃣ Dashboard-da bunları görəcəksən (saxla):

* **Cloud name**
* **API Key**
* **API Secret**

---

#### 🔵 2. Lazım olan paketləri qur (lokalda)

Terminal / VS Code-da:

```bash
pip install cloudinary django-cloudinary-storage
```

Sonra:

```bash
pip freeze > requirements.txt
```

⚠️ **Vacibdir:** `requirements.txt` mütləq GitHub-a push olunmalıdır.

---

#### 🔵 3. `settings.py` dəyişiklikləri (ƏN VACİB HİSSƏ)

#### 📌 `INSTALLED_APPS` içinə əlavə et

```python
INSTALLED_APPS = [
    ...
    'cloudinary',
    'cloudinary_storage',
]
```

---

#### 📌 MEDIA əvəzinə Cloudinary istifadə et

```python
DEFAULT_FILE_STORAGE = 'cloudinary_storage.storage.MediaCloudinaryStorage'
```

👉 `MEDIA_ROOT` və `MEDIA_URL` **tam silinir** (artıq lazım deyil).

---

#### 📌 Cloudinary config (`settings.py` axırına yaxın)

```python
import cloudinary

cloudinary.config(
    cloud_name="CLOUD_NAME",
    api_key="API_KEY",
    api_secret="API_SECRET",
)
```

⚠️ Bunları **öz Cloudinary dashboard dəyərlərinlə** dəyiş.

---

#### 🔵 4. Model (ImageField qalır)

#### `models.py`

```python
from django.db import models

class Product(models.Model):
    title = models.CharField(max_length=200)
    description = models.TextField()
    price = models.DecimalField(max_digits=10, decimal_places=2)
    image = models.ImageField(upload_to="products/")
```

💡 `ImageField` dəyişmir — **sadəcə artıq Cloudinary-yə upload edir**.

---

#### 🔵 5. Migration et (lokalda)

```bash
python manage.py makemigrations
python manage.py migrate
```

---

#### 🔵 6. Admin-də şəkilləri YENİDƏN yüklə

⚠️ **ÇOX VACİB**
Əvvəl əlavə etdiyin şəkillər işləməyəcək.

👉 Django Admin → Product → şəkili **yenidən upload et**

✅ Cloudinary dashboard-da şəkli görməlisən.

---

#### 🔵 7. GitHub-a göndər

```bash
git add .
git commit -m "Add Cloudinary media storage"
git push origin main
```

---

#### 🔵 8. Render-də ENV dəyişənləri əlavə et (TƏHLÜKƏSİZ)

Render Dashboard → **Environment** bölməsi:

| KEY                   | VALUE            |
| --------------------- | ---------------- |
| CLOUDINARY_CLOUD_NAME | sənin cloud name |
| CLOUDINARY_API_KEY    | sənin api key    |
| CLOUDINARY_API_SECRET | sənin api secret |

Sonra `settings.py`-də **təhlükəsiz variant** istifadə et:

```python
import os
import cloudinary

cloudinary.config(
    cloud_name=os.getenv("CLOUDINARY_CLOUD_NAME"),
    api_key=os.getenv("CLOUDINARY_API_KEY"),
    api_secret=os.getenv("CLOUDINARY_API_SECRET"),
)
```

---

#### 🔵 9. Render → Manual Deploy → **Deploy latest commit**

Bir az gözlə… və bitdi 🎉

---

#### ✅ SON NƏTİCƏ

* Admin-də şəkil problemsiz açılır ✔️
* Şəkil linki `https://res.cloudinary.com/...` olur ✔️
* Frontend (GitHub Pages) şəkilləri normal göstərir ✔️
* Render + GitHub Pages problemsiz işləyir ✔️

---

İndi rahat de 👇

* **Admin error verdi?** → buraya at
* **Şəkil API-dən gəlir amma frontend-də görünmür?** → response-a baxaq
* **Cloudinary-də şəkil yoxdur?** → settings-i yoxlayaq

Mən burdayam, qaçmıram 😄👌


### Render.com-dan Admin panel daxil olmaq

https://bazar-backend-5cka.onrender.com/admin/

- username: idrak
- parol: 12345    

### Django admin-ə localdan daxil olmaq

http://127.0.0.1:8000/admin/


- username: idrak
- parol: 12345    

### pythonnawhere


- username: idrak
- parol: 313313Ab_

### Migartion işləri

- python manage.py makemigrations
- python manage.py migrate

### Local-da run

- python manage.py runserver

### API

- http://127.0.0.1:8000/api/products/

### Cloudinary

- https://console.cloudinary.com/app/c-ff504e2efd9031bb218e3f637aa370/settings/api-keys

### Git birinci dəfə

- git init
- git add .
- git commit -m "Initial commit"
- git remote add origin https://github.com/username/bazar-backend.git
- git branch -M main
- git push -u origin main


### Git update üçün

- git add .
- git commit -m "Initial commit"
- git push -u origin main

### pip Freeze etmək

- pip freeze > requirements.txt
